# Datamine SPLIT Process Example

This notebook demonstrates how to configure and run the **`split`** process wrapper in `dmstudio`.

## Process Description

## SPLIT

Take a single input file and copy the data within it to multiple output files.

A new file is generated for every unique value of a selected key field. An optional &**FNAMES** file can be used to define the output filename for each unique value in the * **KEY** field.

If the &**FNAMES** file is not defined then the output filenames will be created from the actual value of the KEY field.

### Input Files:

* **in** (Table):
  Input data file. This file does not have to be sorted on the key field.
  Required=Yes

* **fnames** (Undefined):
  Input file defining the file names to be used for each unique value in the **KEYVALUE**
  field. Must be sorted by **FILENAME** and have the fields; **FILENAME** A - Output file
  name to be created. **KEYVALUE** A/N- Value of the key field which will be written to
  the file defined in FILENAME. This field can be either alphanumeric or numeric but must
  be the same type as the KEY field. A record in the IN file may be written to more than
  one output file by having multiple records with the same **KEYVALUE** and different
  **FILENAMEs** in the **FNAMES** file. Similarly, records with different key values can
  be written to the same file by having multiple records with different **KEYVALUEs** and
  the same **FILENAME**.
  Required=No

### Output Files:

### Fields:

* **key** (Any : IN):
  Alphanumeric or numeric key field in the **IN** file used for selecting data.
  Default=Undefined
  Required=Yes

### Parameters:

* **ndp**:
  Number of decimal places to be used for creating file names from a numeric **KEY**
  field (0). Options: 0: do not display any digits after the decimal point. >0= replace
  the decimal point by the letter 'D' and display the specified number of digits following
  it. -1= create a file name using scientific notation.
  Range=-1,1
  Values=-1,0,1
  Default=0
  Required=No

* **maxfiles**:
  Maximum number of files which will be created (50). If this maximum is exceeded an
  error message is printed and the process is terminated. The maximum allowed value is 250
  files. Rules for creating file names from **KEY** field; If KEY field is alphanumeric:
  \- only the first 24 characters will be used. \- if a blank is encountered in the
  **KEY** value only those characters preceding the blank will be used. \- if a decimal
  point is encountered only the character immediately following the decimal point will be
  used. If KEY field is numeric: \- all file names are preceeded by one of two letters; P
  = plus M = minus \- file names that are too long due to a large number of decimal places
  will use only the first eight characters. \- all numbers with more than 7 digits
  preceeding the decimal point will be converted to scientific notation (for example,
  1.23456E+09 becomes P1235P06) - trace (TR) are written to the file TRACE. \- maximum
  values (+) are written to PLUS. \- undefined values (-) are written to MINUS.
  Range=Undefined
  Values=Undefined
  Default=50
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('split')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute split
print("Running split...")
dm_cmd.split(
    in_i='t_assays',  # required
    key_f=['BHID'],  # required
    # fnames_i='optional',  # optional
    # ndp_p=0,  # optional
    # maxfiles_p=50,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("split execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_split_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")